# **LlamaIndex: Conversational AI with Chat Engines**  

To make **Retrieval-Augmented Generation (RAG)** interactive, we extend our **retrieval and response pipeline** with a **chat engine** for multi-turn conversations. This allows the system to **remember interactions**, rewrite queries, and generate more context-aware responses.  

### This notebook covers:
- Setting up a **chat engine**  
- Exploring **multi-turn conversation modes**  
- Configuring **custom prompts and chat history**  
- Enabling **streaming responses** for real-time interaction  

## **Understanding Chat Engines in LlamaIndex**  

A **chat engine** enhances retrieval by:  
1. **Tracking conversation history** instead of treating queries in isolation.  
2. **Rewriting follow-up questions** for better context.  
3. **Retrieving relevant nodes** and appending them to queries.  
4. **Generating informed responses** using an LLM.  

By structuring interactions, **chat engines improve response quality** for complex queries and dynamic conversations.  

## Setup: Preparing the Index for Chats  

First, we restore the index from previous notebooks.

In [15]:
%load_ext ipython_async

In [ ]:
%%asynccmd
mkdir .\vectorstore\postgres &
initdb -D .\vectorstore\postgres --encoding=UTF8 --locale=en_US.UTF-8 &
pg_ctl -D .\vectorstore\postgres -l logfile start

A subdirectory or file .\vectorstore\postgres already exists.
The files belonging to this database system will be owned by user "acisse".
This user must also own the server process.

The database cluster will be initialized with locale "en_US.UTF-8".
The default text search configuration will be set to "english".

Data page checksums are disabled.

initdb: error: directory "vectorstore/postgres" exists but is not empty
initdb: hint: If you want to create a new database system, either remove or empty the directory "vectorstore/postgres" or run initdb with an argument other than "vectorstore/postgres".
waiting for server to start.... done
server started


<Thread(Thread-36 (run_command), started daemon 37076)>

In [17]:
import os

In [18]:
db_name = "vector_db"
user = "acisse"

To restore the index, we need to restore the vector store and embedding model first:

In [19]:
from llama_index.vector_stores.postgres import PGVectorStore

vector_store = PGVectorStore.from_params(
    database=db_name,
    host="localhost",
    password=os.getenv("PSQL_PASSWORD"),
    port=5432,
    user=user,
    table_name="physics_docs",
    embed_dim=384,
    hybrid_search=True
)

In [20]:
from llama_index.embeddings.langchain import LangchainEmbedding
from langchain.embeddings import HuggingFaceEmbeddings

# Load MiniLM embedding model (fast & efficient)
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Wrap the model for LlamaIndex
embed_model = LangchainEmbedding(embedding_model)

Finally, we can restore the index:

In [21]:
from pathlib import Path
from llama_index.core import StorageContext, load_index_from_storage

# Load the stored index
index_save_path = Path("assets", "llama_vector_search", "physics_documents_index")
storage_context = StorageContext.from_defaults(persist_dir=index_save_path, vector_store=vector_store)
index = load_index_from_storage(storage_context, embed_model=embed_model)

## Interacting with the Chat Engine API in LlamaIndex

LlamaIndex provides a chat engine abstraction that allows users to interact with retrieval-augmented generation (RAG) systems in a multi-turn conversational format. Unlike a standard query engine, which processes each query independently, a chat engine maintains chat history, dynamically reformulates follow-up questions, and retrieves relevant context before generating responses.

### How to Interact with the Chat Engine API

LlamaIndex provides multiple ways to interact with a chat engine:  

In [22]:
from dotenv import load_dotenv
load_dotenv()

True

In [23]:

from llama_index.llms.openai import OpenAI
from utils import display_citation

llm_openai=OpenAI(model="gpt-4o")
chat_engine = index.as_chat_engine(llm=llm_openai)
response = chat_engine.chat("Tell me a joke.")
display_citation(response.response)

> Why don't scientists trust atoms? Because they make up everything!

To reset the conversation history, use:

In [24]:
chat_engine.reset()

For an interactive chat session, run:

In [25]:
response = chat_engine.chat("What joke did you just tell me?")
display_citation(response.response)

> I haven't told you a joke yet. Would you like to hear one?

This starts a command-line chat interface, allowing for a dynamic back-and-forth conversation.

In [26]:
chat_engine.chat_repl()

===== Entering Chat REPL =====
Type "exit" to exit.

Assistant: Hello! How can I assist you today?



# Understanding Chat Modes in LlamaIndex

LlamaIndex provides different chat modes to process user input, retrieve context, and generate responses based on specific needs.

### `condense_question` Mode  
Rewrites follow-up queries into standalone questions for better retrieval.

Useful when:
- Users ask ambiguous follow-ups (e.g., "What about his books?")  
- Ensuring each query is **independent** and **retrieval-friendly**  

In [30]:
chat_engine = index.as_chat_engine(chat_mode="condense_question", verbose=True)
response = chat_engine.chat("Who is Juan Maldacena.")
display_citation(response.response)

Querying with: Who is Juan Maldacena.


> Juan Maldacena is an Argentine theoretical physicist known for his significant contributions to the foundations of string theory and quantum gravity. He is the Carl P. Feinberg Professor in the School of Natural Sciences at the Institute for Advanced Study in Princeton. Maldacena is most famous for his discovery of the AdS/CFT correspondence, which is a realization of the holographic principle in string theory.

In [31]:
response = chat_engine.chat("What is the AdS/CFT correspondence.")
display_citation(response.response)

Querying with: What is the AdS/CFT correspondence and who is Juan Maldacena?


> The AdS/CFT correspondence is a realization of the holographic principle in string theory, proposing the equivalence of certain theories of quantum gravity to other quantum mechanical theories in one fewer spacetime dimensions. Juan Martín Maldacena is an Argentine theoretical physicist known for his significant contributions to the foundations of string theory and quantum gravity, with his most famous discovery being the AdS/CFT correspondence.

`context`  
This mode retrieves relevant documents from the index and injects them directly into the system prompt, allowing the LLM to generate responses with more **grounded context**.  

In [33]:
chat_engine = index.as_chat_engine(chat_mode="context", verbose=True)
response = chat_engine.chat("Tell me about supersymmetry.")
display_citation(response.response)

> Supersymmetry is a fundamental concept in theoretical physics that relates particles with different spin quantum numbers. It is a symmetry that relates bosons (particles with integer spin) and fermions (particles with half-integer spin). In theories that incorporate supersymmetry, each particle has a superpartner with different spin.
> 
> Supersymmetry plays a crucial role in many areas of theoretical physics, including particle physics, string theory, and quantum field theory. It provides a framework for addressing some of the limitations of the Standard Model of particle physics and offers solutions to various theoretical problems.
> 
> In the context of the text you provided, supersymmetry is discussed in the context of supergravity solutions and field theories. The presence of supersymmetry in these theories can lead to more elegant and mathematically consistent descriptions of physical phenomena.

`condense_plus_context`  
This mode combines query rewriting (like `condense_question`) with context injection (like `context`). It ensures follow-up questions are rewritten while also retrieving relevant documents.  

In [35]:
chat_engine = index.as_chat_engine(chat_mode="condense_plus_context", verbose=True)
response = chat_engine.chat("What is supergravity?")
display_citation(response.response)

Condensed question: What is supergravity.


> Supergravity is a field theory that combines the principles of supersymmetry and general relativity. It is a theory that extends Einstein's theory of general relativity to include supersymmetry, which is a symmetry that relates particles with different spin quantum numbers. Supergravity theories aim to describe the gravitational interactions of supersymmetric particles. In the context of M-theory, supergravity plays a crucial role as it provides a framework for understanding the dynamics of the theory in higher dimensions. The study of supergravity is essential in theoretical physics, especially in the context of string theory and M-theory, where it helps in understanding the fundamental forces of nature and the structure of spacetime.

In [37]:
response = chat_engine.chat("What are world sheets?")
display_citation(response.response)

Condensed question: What role do world sheets play in the context of supergravity theories?


> World sheets are two-dimensional surfaces that play a crucial role in string theory. In the context of string theory, particles are replaced by one-dimensional strings. The motion of these strings through spacetime is described by world sheets, which are two-dimensional surfaces that represent the trajectory of a string as it moves through spacetime.
> 
> World sheets provide a way to visualize and study the behavior of strings in different spacetime backgrounds. The dynamics of strings, including their vibrations and interactions, are encoded in the geometry of the world sheet. By analyzing the properties of world sheets, researchers can understand the fundamental principles of string theory and how strings interact with the spacetime they propagate through.
> 
> In supergravity theories, world sheets are particularly important as they are used to define supersymmetric interacting theories arising from specific gravity backgrounds. The actions on the world sheet are derived from the gravity backgrounds, and the solutions are described in terms of the metric and field strengths on the world sheet.
> 
> Overall, world sheets are fundamental constructs in string theory and supergravity, providing a geometric representation of the dynamics of strings and serving as a key tool for understanding the behavior of strings in different spacetime environments.

`simple`
This mode allows direct interaction with the LLM without retrieval. The model generates responses based only on its internal knowledge.  


In [38]:
chat_engine = index.as_chat_engine(chat_mode="simple", verbose=True)
response = chat_engine.chat("What is supergravity?")
display_citation(response.response)

> Supergravity is a theory that combines the principles of general relativity (which describes the force of gravity) with those of supersymmetry (which proposes a symmetry between particles with different spin quantum numbers). It is a theoretical framework that aims to unify the four fundamental forces of nature (gravity, electromagnetism, and the strong and weak nuclear forces) into a single, coherent theory. Supergravity is a key component of string theory, which is a theoretical framework that attempts to describe the fundamental particles and forces of the universe in a unified manner.

We can note that no relation to M-theory was made, compared to the previous query, since supersymmetry is addressed in the context of M-theory in our dataset.

In [39]:
response = chat_engine.chat("What are world sheets?")
display_citation(response.response)

> In physics, particularly in the context of string theory, a world sheet is a two-dimensional surface that represents the trajectory of a string moving through spacetime. The concept of a world sheet is used to describe the behavior of strings in string theory, which posits that fundamental particles are not point-like particles but rather tiny, vibrating strings.
> 
> The world sheet of a string is a two-dimensional surface that extends through time (representing the evolution of the string over time) and space (representing the spatial dimensions in which the string moves). The dynamics of the string are described by the interactions of the string with its world sheet, and the properties of the string, such as its mass, charge, and spin, are determined by the geometry and topology of the world sheet.
> 
> In string theory, the interactions and behavior of strings are described by the dynamics of their world sheets, making the concept of world sheets a fundamental aspect of the theory.

Here supergravity isn't mentioned, hinting at the missing of retrieval.

`react`  
This mode enables ReAct-style reasoning, where the system retrieves facts, reasons step-by-step, and then answers It is ideal for complex, multi-step problem-solving.  

In [40]:
chat_engine = index.as_chat_engine(chat_mode="react", verbose=True)
response = chat_engine.chat("Explain the holographic principle so that it is understandable for a child.")
display_citation(response.response)

Added user message to memory: Explain the holographic principle so that it is understandable for a child.
=== Calling Function ===
Calling function: query_engine_tool with args: {"input":"Explain the holographic principle in simple terms for a child."}
Got output: The holographic principle is like a magical mirror that shows us two different worlds that are actually the same. Imagine one world where everything is made of tiny strings and another world where there are no strings, just regular things like toys and books. The holographic principle says that these two worlds are connected in a special way, like a secret code that helps us understand both worlds better. It's like having a special pair of glasses that let you see two different pictures at the same time, even though they look different, they are actually related in a very cool and mysterious way.



> The holographic principle is like a magical mirror that shows us two different worlds that are actually the same. Imagine one world where everything is made of tiny strings and another world where there are no strings, just regular things like toys and books. The holographic principle says that these two worlds are connected in a special way, like a secret code that helps us understand both worlds better. It's like having a special pair of glasses that let you see two different pictures at the same time, even though they look different, they are actually related in a very cool and mysterious way.

In [41]:
chat_engine = index.as_chat_engine(chat_mode="react", verbose=True)
response = chat_engine.chat("How does the holographic principle relate to string theory,"
                            " supersymmetry, supergravity and worldsheets?")
display_citation(response.response)

Added user message to memory: How does the holographic principle relate to string theory, supersymmetry, supergravity and worldsheets?
=== Calling Function ===
Calling function: query_engine_tool with args: {"input": "How does the holographic principle relate to string theory?"}
Got output: The holographic principle is realized through the AdS/CFT correspondence, which is a conjecture about the equivalence of string theory on Anti-de Sitter (AdS) space and a conformal field theory defined on the boundary of the AdS space. This correspondence suggests that certain theories of quantum gravity are equivalent to other quantum mechanical theories in one fewer spacetime dimensions.

=== Calling Function ===
Calling function: query_engine_tool with args: {"input": "How does the holographic principle relate to supersymmetry?"}
Got output: The holographic principle is related to supersymmetry through the AdS/CFT correspondence, which is a conjecture about the equivalence of string theory on Ant

> The holographic principle is related to various concepts in theoretical physics:
> 
> 1. **String Theory**: The holographic principle is realized through the AdS/CFT correspondence, which suggests the equivalence of string theory on Anti-de Sitter (AdS) space and a conformal field theory defined on the boundary of the AdS space. This correspondence demonstrates a deep connection between gravitational theories and quantum mechanical theories in one fewer spacetime dimensions.
> 
> 2. **Supersymmetry**: The holographic principle is connected to supersymmetry through the AdS/CFT correspondence. This correspondence showcases the equivalence of certain theories of quantum gravity to other quantum mechanical theories in fewer spacetime dimensions, highlighting the relationship between gravitational theories and quantum mechanical theories without gravitational force.
> 
> 3. **Supergravity**: The holographic principle is realized through the AdS/CFT correspondence, providing a non-perturbative formulation of string theory. This correspondence demonstrates the equivalence of certain theories of quantum gravity to other quantum mechanical theories in fewer spacetime dimensions, emphasizing the connection between gravity and quantum mechanics.
> 
> 4. **Worldsheets**: The holographic principle is related to worldsheets through the AdS/CFT correspondence. This correspondence allows for the description of physical observables defined in one theory to be described in the equivalent theory, illustrating the connection between the holographic principle and worldsheets in theoretical physics.

`openai`
This mode leverages OpenAI’s function-calling API allowing the chat engine to call external tools and return structured responses.  

In [42]:
chat_engine = index.as_chat_engine(chat_mode="openai", verbose=True)
response = chat_engine.chat("What's the weather in New York?")
display_citation(response.response)

Added user message to memory: What's the weather in New York?
=== Calling Function ===
Calling function: query_engine_tool with args: {"input":"What's the weather in New York?"}
Got output: I'm unable to provide real-time information such as the current weather in New York.



> I'm unable to provide real-time information such as the current weather in New York.